# Travel cost and accessibility matrices

This notebook prepares the OD travel-cost inputs used in the destination choice model. The goal is to transform the original MOBi5 skim matrices into matrices that are aligned with the selected 2023 traffic zones, convert them into consistent units, and derive accessibility-related indicators such as mode utilities and the expected maximum utility (EMU).

The main output is a set of OMX matrices stored in:

`TravelCost/skims/MOBi5_2023_ready/`

These matrices are later used to attach travel impedance and accessibility information to the observed origin–destination leisure trips and to the sampled non-chosen alternatives.

### Chunk 1: Preparing 2023-ready MOBi5 skim matrices

The first chunk reads the original MOBi5 car and public transport skim matrices and reduces them to the set of zones available in the 2023 accessibility layer:

`TravelCost/accessibility/2023/acc_2023.gpkg`

The original MOBi5 matrices contain a larger set of zones. Therefore, the script checks which zones are also present in the 2023 accessibility dataset and keeps only those. The zone order is preserved using the original MOBi5 `NO` mapping, so that all output matrices remain consistently indexed.

The script writes three ready-to-use OMX files:

- `car_skims_2023_ready.omx`
- `pt_skims_2023_ready.omx`
- `beeline_2023_ready.omx`

For the car and PT matrices, the script also converts the units:

- distances: metres → kilometres
- travel times: seconds → minutes

The beeline matrix is computed directly from the centroids of the selected zones and stores straight-line distances in kilometres.

### Chunk 2: Computing 2023 mode utilities and EMU

The second chunk computes mode-specific OD utilities for:

- walking
- cycling
- car
- public transport

The input matrices are the ready car and PT skims created in Chunk 1. In addition, the script uses accessibility attributes from:

`TravelCost/accessibility/2023/acc_2023.gpkg`

In particular, it uses:

- `at_car`: car access time
- `pc_car`: parking cost

The car utility combines travel time, distance bands, access time, and parking cost. The public transport utility combines in-vehicle time, access and egress time, frequency/headway, transfers, and the train travel-time share. Walking and cycling utilities are derived from distance-based impedance.

After computing the four mode utilities, the script calculates the expected maximum utility:

`utility_emu`

This is computed as a logsum over the four available mode utilities. The resulting file is:

`utilities_2023_ready.omx`

This matrix is the main accessibility/impedance input used later in the destination choice model.

### Chunk 3: Checking the 2023 utility matrices

The third chunk performs diagnostic checks on the output utility matrices.

It reports:

- available matrices in the OMX file
- mapping consistency of the `NO` zone index
- basic statistics for each utility matrix
- NaN counts
- sample-based correlations between EMU and beeline distance

The purpose is to verify that the matrices were written correctly and that the EMU behaves plausibly with respect to distance. Since accessibility should generally decrease as distance increases, a negative relationship between EMU and beeline distance is expected.

### Chunk 4: Computing 2017 utilities for comparison

The fourth chunk repeats the utility computation using the 2017 accessibility attributes:

`TravelCost/accessibility/2017/acc_2017.gpkg`

The same ready MOBi5 skim matrices are used, but the car access time and parking cost terms are taken from the 2017 accessibility layer. This produces:

`utilities_2017_ready.omx`

The second part of the chunk compares:

- EMU 2017 vs EMU 2023
- EMU 2017 vs beeline distance
- EMU 2017 vs log-transformed beeline distance

This is used as a robustness and consistency check between the two accessibility years.

### Chunk 5: Trip-level correlation checks

The fifth chunk works with the extracted leisure trip tables after EMU values have been attached to the observed OD pairs.

It reads:

- `destinations_YS_emu.csv`
- `destinations_OS_emu.csv`
- `destinations_YL_emu.csv`
- `destinations_OL_emu.csv`

The four files correspond to the four demand segments:

- young short trips
- old short trips
- young long/day trips
- old long/day trips

Trips with `leisure_cat == "NK"` are excluded because their leisure subtype is unavailable or not usable for category-specific interpretation.

The script then checks the correlation between observed trip distance and the EMU indicators:

- `dist_km` vs `emu_2017`
- `dist_km` vs `emu_2023`
- `log1p(dist_km)` vs `emu_2017`
- `log1p(dist_km)` vs `emu_2023`

This provides a trip-level plausibility check of the accessibility variables.

### Chunk 6: Removing Liechtenstein zones

The sixth chunk creates versions of the EMU and beeline matrices without Liechtenstein zones.

The original matrices include a small set of Liechtenstein NPVM zones. Since the thesis analysis focuses on Swiss destination choice, these zones are removed from the mapping and from all square OD matrices.

The outputs are:

- `utilities_2023_ready_woLIE.omx`
- `beeline_2023_ready_woLIE.omx`

Rows and columns corresponding to Liechtenstein zones are dropped, and the `NO` mapping is rewritten accordingly.

### Chunk 7: Creating an average car distance matrix

The seventh chunk creates a single average car distance matrix from the AM and PM car distance skims.

It reads:

- `am_distances_ready`
- `pm_distances_ready`

from:

`car_skims_2023_ready.omx`

and writes their average to:

`distance_avg_2023_ready.omx`

with matrix name:

`avg_distances_ready`

This provides a simpler OD distance matrix that can be used when a single representative car distance measure is needed.

### Chunk 8: Correlating accessibility with destination attractivity variables

The final chunk compares the destination-side accessibility measure with the final attractivity predictors used in the thesis model.

It reads the selected traffic-zone feature layer:

`VariableAnalysis/TZ_first_sel_log1p.gpkg`

and the EMU matrix:

`utilities_2023_ready.omx`

For each destination zone, the script computes destination-side EMU aggregates from the OD matrix. The main accessibility indicator used here is:

`emu_logsumexp_i`

This represents how accessible each destination is from all origins, aggregated through a logsumexp transformation.

The script then correlates this destination-side accessibility indicator with the final 13 attractivity variables used in the thesis. The output table is saved as:

`ModelRuns/BaselineFinal2/corr_accessibility_vs_factors.csv`

This check is useful to understand whether the attractivity variables are strongly related to general accessibility. It helps interpret the model specification and identify potential overlap between destination attractiveness and accessibility.

## Chunk 1

In [ ]:
import numpy as np
import geopandas as gpd
from pathlib import Path
import openmatrix as omx
import tables as tb

# ============================================================
# CONFIG
# ============================================================
ACC2023_GPKG  = Path("TravelCost/accessibility/2023/acc_2023.gpkg")
ACC2023_LAYER = "acc_2023"

MOBI5_CAR = Path("TravelCost/skims/MOBi5/car_skims.omx")
MOBI5_PT  = Path("TravelCost/skims/MOBi5/pt_skims.omx")

OUT_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CAR     = OUT_DIR / "car_skims_2023_ready.omx"
OUT_PT      = OUT_DIR / "pt_skims_2023_ready.omx"
OUT_BEELINE = OUT_DIR / "beeline_2023_ready.omx"

MAPPING_NAME   = "NO"
CHUNK_ROWS     = 200
BEELINE_CHUNK  = 200

CAR_MATS = ["am_distances", "am_travel_times", "pm_distances", "pm_travel_times"]
PT_MATS  = [
    "access_times", "adaption_times", "data_counts", "distances", "egress_times",
    "frequencies", "train_distance_shares", "train_traveltime_shares",
    "transfer_counts", "travel_times"
]

# unit conversion sets
DIST_MATS = {"am_distances", "pm_distances", "distances"}  # metres -> km
TIME_MATS = {"am_travel_times", "pm_travel_times", "access_times", "adaption_times", "egress_times", "travel_times"}  # seconds -> minutes

# ============================================================
# Helpers
# ============================================================
def read_no_dict(omx_path: Path, mapping_name="NO"):
    """Read NO from original MOBi5 files: expected dict zone_id -> idx."""
    f = omx.open_file(str(omx_path), "r")
    if mapping_name not in list(f.list_mappings()):
        f.close()
        raise ValueError(f"Mapping '{mapping_name}' not found in {omx_path.name}")
    no = dict(f.mapping(mapping_name))  # zone_id -> idx
    f.close()
    return no

def read_mapping_array_safe(f, name="NO"):
    """
    Robustly read an OMX mapping as a 1D numpy array (idx -> zone_id),
    regardless of whether it supports slicing.
    """
    m = f.mapping(name)

    # Case 1: tables.Array-like with slicing
    try:
        arr = np.asarray(m[:])
        if arr.ndim == 1:
            return arr
    except Exception:
        pass

    # Case 2: iterable
    try:
        arr = np.asarray(list(m))
        if arr.ndim == 1:
            return arr
    except Exception:
        pass

    # Case 3: numpy array conversion
    arr = np.asarray(m)
    if arr.ndim == 0:
        raise ValueError("Mapping is scalar/unsized; not an array mapping.")
    return arr

def explore_original_mobi5(omx_path: Path, mats_keep: list, acc_ids_set: set, mapping_name="NO", n_examples=10):
    print("\n" + "="*90)
    print("ORIGINAL OMX:", omx_path)
    print("="*90)

    f = omx.open_file(str(omx_path), "r")
    mats = list(f.list_matrices())
    maps = list(f.list_mappings())

    print("\nMatrices:", len(mats))
    for m in mats:
        M = f[m]
        flag = "KEEP" if m in mats_keep else "skip"
        print(f" - {m:25s} shape={tuple(M.shape)} dtype={M.dtype} [{flag}]")

    print("\nMappings:", maps)

    if mapping_name not in maps:
        f.close()
        raise ValueError(f"Mapping '{mapping_name}' not found in {omx_path.name}")

    no = dict(f.mapping(mapping_name))
    keys = np.array(list(no.keys()), dtype=np.int64)
    vals = np.array(list(no.values()), dtype=np.int64)

    print(f"\nMapping '{mapping_name}' summary:")
    print(" - entries:", len(no))
    print(" - idx range:", int(vals.min()), "to", int(vals.max()))
    print(" - first examples zone_id -> idx:")
    for z in keys[:n_examples]:
        print(f"    {int(z)} -> {int(no[int(z)])}")

    keyset = set(keys.tolist())
    miss = sorted(list(acc_ids_set - keyset))
    print("\nCoverage wrt acc_2023:")
    print(" - acc size:", len(acc_ids_set))
    print(" - in mapping:", len(acc_ids_set) - len(miss))
    print(" - missing:", len(miss))
    if len(miss) > 0:
        print("   missing examples:", miss[:15])
    print(" - extra ids in mapping (not in acc):", len(keyset - acc_ids_set))

    # unit sanity: sample scalar reads from first kept matrix that exists
    m0 = None
    for cand in mats_keep:
        if cand in mats:
            m0 = cand
            break
    if m0 is not None:
        M0 = f[m0]
        n = int(M0.shape[0])
        rng = np.random.default_rng(0)
        ii = rng.integers(0, n, size=2000, dtype=np.int64)
        jj = rng.integers(0, n, size=2000, dtype=np.int64)
        samp = np.array([float(M0[int(i), int(j)]) for i, j in zip(ii, jj)], dtype=np.float64)
        samp = samp[np.isfinite(samp)]
        print(f"\nUnit sanity (sample from '{m0}'):")
        if samp.size:
            print(" - median:", float(np.nanmedian(samp)), "p95:", float(np.nanpercentile(samp, 95)))
            print(" - if these are large (~1e5 or ~1e4): they are metres/seconds and must be converted")
        else:
            print(" - no finite values sampled")

    f.close()
    return no

def write_ready_omx(in_omx: Path, out_omx: Path, matrices: list, keep_zone_ids: list,
                    row_mask: np.ndarray, col_mask: np.ndarray, out_pos: np.ndarray):
    """
    Writes reduced KxK matrices + mapping NO as ARRAY (idx->zone_id).
    Safe read later: arr = f.mapping('NO')[:]
    """
    if out_omx.exists():
        out_omx.unlink()

    fin = omx.open_file(str(in_omx), "r")
    fout = omx.open_file(str(out_omx), "w")

    # mapping as array: output index -> zone_id
    fout.create_mapping(MAPPING_NAME, np.array(keep_zone_ids, dtype=np.int64))
    print(f"\n✅ Mapping written to {out_omx.name}: NO as array (idx->zone_id), len={len(keep_zone_ids)}")

    atom_f4 = tb.Float32Atom()

    K = len(keep_zone_ids)
    for name in matrices:
        if name not in fin.list_matrices():
            fin.close(); fout.close()
            raise ValueError(f"Matrix '{name}' not found in {in_omx.name}")

        Min = fin[name]
        n = int(Min.shape[0])

        out_name = f"{name}_ready"
        fout.create_matrix(out_name, atom=atom_f4, shape=(K, K))
        Mout = fout[out_name]

        print(f"[{out_omx.name}] building {out_name} from {name} ...")

        for r0 in range(0, n, CHUNK_ROWS):
            r1 = min(n, r0 + CHUNK_ROWS)

            block = np.asarray(Min[r0:r1, :], dtype=np.float32)  # (chunk, n)

            keep_local = row_mask[r0:r1]
            if not np.any(keep_local):
                continue

            block = block[keep_local, :]
            block = block[:, col_mask]

            # clean
            block[np.isinf(block)] = np.nan

            # convert units
            if name in DIST_MATS:
                block = block / 1000.0
            if name in TIME_MATS:
                block = block / 60.0

            orig_rows = np.arange(r0, r1, dtype=np.int64)[keep_local]
            out_rows = out_pos[orig_rows]

            # Since keep_zone_ids are ordered by original index, out_rows in this chunk are contiguous
            out_start = int(out_rows[0])
            out_end   = int(out_rows[-1] + 1)

            Mout[out_start:out_end, :] = block

    fin.close()
    fout.close()
    print(f"✅ Finished writing {out_omx}")

def build_centroids_from_gdf(gdf: gpd.GeoDataFrame, id_col="zone_id"):
    """
    Build centroid dict {zone_id: shapely Point} from a GeoDataFrame.
    If geometries are polygons -> centroid; if points -> use as is.
    """
    if id_col not in gdf.columns:
        raise ValueError(f"'{id_col}' not found in GeoDataFrame columns: {list(gdf.columns)}")
    if gdf.geometry is None:
        raise ValueError("GeoDataFrame has no geometry column.")
    if gdf.geometry.is_empty.any():
        gdf = gdf[~gdf.geometry.is_empty].copy()

    # ensure geometry is in a metric CRS (EPSG:2056 as you use)
    if gdf.crs is None:
        raise ValueError("Input GeoDataFrame has no CRS. Please set it before computing distances.")
    if int(gdf.crs.to_epsg() or 0) != 2056:
        # reproject for metric distances
        gdf = gdf.to_crs(epsg=2056)

    # compute centroids
    geom = gdf.geometry
    # if already points, centroid == itself; for polygons it becomes centroid
    cents = geom.centroid
    ids = gdf[id_col].astype(int).to_numpy()
    return {int(z): pt for z, pt in zip(ids, cents)}

def write_beeline_omx(out_omx: Path, keep_zone_ids: list, centroids: dict):
    if out_omx.exists():
        out_omx.unlink()

    fout = omx.open_file(str(out_omx), "w")
    fout.create_mapping(MAPPING_NAME, np.array(keep_zone_ids, dtype=np.int64))
    print(f"\n✅ Mapping written to {out_omx.name}: NO as array (idx->zone_id), len={len(keep_zone_ids)}")

    atom_f4 = tb.Float32Atom()
    K = len(keep_zone_ids)
    fout.create_matrix("beeline_km_ready", atom=atom_f4, shape=(K, K))
    Mout = fout["beeline_km_ready"]

    # coordinates in output order
    xs = np.empty(K, dtype=np.float64)
    ys = np.empty(K, dtype=np.float64)
    for i, z in enumerate(keep_zone_ids):
        pt = centroids.get(int(z))
        if pt is None:
            fout.close()
            raise ValueError(f"Missing centroid for zone_id={z}")
        xs[i] = pt.x
        ys[i] = pt.y

    print("Writing beeline_km_ready in chunks...")
    for r0 in range(0, K, BEELINE_CHUNK):
        r1 = min(K, r0 + BEELINE_CHUNK)
        xb = xs[r0:r1][:, None]
        yb = ys[r0:r1][:, None]
        dx = xb - xs[None, :]
        dy = yb - ys[None, :]
        dist_km = np.sqrt(dx*dx + dy*dy) / 1000.0
        Mout[r0:r1, :] = dist_km.astype(np.float32)
        if r0 % (BEELINE_CHUNK * 10) == 0:
            print(f"  rows {r0}..{r1-1} written")

    fout.close()
    print(f"✅ Finished writing {out_omx}")

def check_ready_file(path, sample_mats):
    f = omx.open_file(str(path), "r")
    print("\n" + "="*80)
    print("CHECK READY:", path.name)
    print("="*80)
    print("Matrices:", list(f.list_matrices()))
    print("Mappings:", list(f.list_mappings()))

    no_arr = read_mapping_array_safe(f, "NO").astype(int)
    print("NO len:", len(no_arr), "first:", no_arr[:3].tolist(), "last:", no_arr[-3:].tolist())

    for mn in sample_mats:
        if mn in f.list_matrices():
            M = f[mn]
            print(f"{mn}: M[0,0]={float(M[0,0]):.3f}  M[0,1]={float(M[0,1]):.3f}  M[1,0]={float(M[1,0]):.3f}")

    f.close()

# ============================================================
# RUN
# ============================================================

# --- Load acc_2023 layer (must contain zone_id + geometry) ---
acc23 = gpd.read_file(ACC2023_GPKG, layer=ACC2023_LAYER)

if "zone_id" not in acc23.columns:
    raise ValueError(f"'zone_id' column not found in {ACC2023_GPKG} layer {ACC2023_LAYER}. "
                     f"Columns are: {list(acc23.columns)}")

if acc23.geometry is None:
    raise ValueError(f"Layer {ACC2023_LAYER} has no geometry. "
                     "I need zone geometries (or at least zone points) to compute beeline distances.")

# ensure CRS is metric for centroid distance (EPSG:2056 preferred)
if acc23.crs is None:
    # If you know it's LV95, you can set it here; otherwise better to raise
    raise ValueError("acc23 has no CRS. Please set acc23.crs (expected EPSG:2056) before proceeding.")
if int(acc23.crs.to_epsg() or 0) != 2056:
    acc23 = acc23.to_crs(epsg=2056)

# --- zone id set from acc23 ---
ACC_IDS_SET = set(acc23["zone_id"].astype(int).unique().tolist())

# --- build centroid dict from acc23 geometries ---
CENTROIDS = build_centroids_from_gdf(acc23, id_col="zone_id")
print("✅ CENTROIDS built:", len(CENTROIDS))

# Exploration original
no_car = explore_original_mobi5(MOBI5_CAR, CAR_MATS, ACC_IDS_SET, mapping_name=MAPPING_NAME)
no_pt  = explore_original_mobi5(MOBI5_PT,  PT_MATS,  ACC_IDS_SET, mapping_name=MAPPING_NAME)

print("\nNO key sets identical (car vs pt)?", set(no_car.keys()) == set(no_pt.keys()))

# Keep zones: order by original MOBi5 index (car mapping used as reference)
keep_zone_ids = sorted([z for z in ACC_IDS_SET if z in no_car], key=lambda z: no_car[z])
K = len(keep_zone_ids)
print("\nKEEP zones for 2023-ready:", K)

# sanity: centroid coverage for keep zones
missing_centroids = [z for z in keep_zone_ids if int(z) not in CENTROIDS]
if missing_centroids:
    raise ValueError(f"Missing centroids for {len(missing_centroids)} keep zones. Examples: {missing_centroids[:10]}")

N = max(no_car.values()) + 1  # original matrix size, e.g. 8688
keep_idx = np.array([no_car[z] for z in keep_zone_ids], dtype=np.int64)

row_mask = np.zeros(N, dtype=bool); row_mask[keep_idx] = True
col_mask = row_mask.copy()

out_pos = np.full(N, -1, dtype=np.int64)
out_pos[keep_idx] = np.arange(K, dtype=np.int64)

# Write ready matrices
write_ready_omx(MOBI5_CAR, OUT_CAR, CAR_MATS, keep_zone_ids, row_mask, col_mask, out_pos)
write_ready_omx(MOBI5_PT,  OUT_PT,  PT_MATS,  keep_zone_ids, row_mask, col_mask, out_pos)

# Write beeline full matrix (km)
write_beeline_omx(OUT_BEELINE, keep_zone_ids, CENTROIDS)

# Final checks
check_ready_file(OUT_CAR, ["am_distances_ready", "am_travel_times_ready"])
check_ready_file(OUT_PT,  ["access_times_ready", "travel_times_ready"])
check_ready_file(OUT_BEELINE, ["beeline_km_ready"])

print("\n✅ Outputs:")
print(" -", OUT_CAR)
print(" -", OUT_PT)
print(" -", OUT_BEELINE)


## Chunk 2

In [ ]:
import numpy as np
import geopandas as gpd
from pathlib import Path
import openmatrix as omx
import tables as tb

# ------------------------------------------------------------
# PATHS (adjust if needed)
# ------------------------------------------------------------
READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
CAR_READY = READY_DIR / "car_skims_2023_ready.omx"
PT_READY  = READY_DIR / "pt_skims_2023_ready.omx"

ACC2023_GPKG  = Path("TravelCost/accessibility/2023/acc_2023.gpkg")
ACC2023_LAYER = "acc_2023"

OUT_UTIL = READY_DIR / "utilities_2023_ready.omx"

MAPPING_NAME = "NO"
CHUNK_ROWS   = 200  # you can increase if RAM allows

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def read_no_array_safe(f, name="NO"):
    """
    Read mapping NO as 1D array idx->zone_id.
    Works even if the underlying object doesn't support slicing cleanly.
    """
    m = f.mapping(name)
    try:
        arr = np.asarray(m[:])
        if arr.ndim == 1:
            return arr.astype(int)
    except Exception:
        pass
    try:
        arr = np.asarray(list(m))
        if arr.ndim == 1:
            return arr.astype(int)
    except Exception:
        pass
    arr = np.asarray(m)
    if arr.ndim == 0:
        raise ValueError("Mapping NO is scalar/unsized. Not expected for ready files.")
    return arr.astype(int)

def dist_band_0015(d):
    return np.minimum(d, 15.0)

def dist_band_1550(d):
    return np.maximum(0.0, np.minimum(d - 15.0, 35.0))

def dist_band_5099(d):
    return np.maximum(0.0, np.minimum(d - 50.0, 50.0))

def dist_band_100x(d):
    return np.maximum(0.0, d - 100.0)

def logsumexp_4(U1, U2, U3, U4, theta=1.0):
    """
    Stable logsumexp with NaN handling:
    - NaN -> treated as -inf (mode unavailable)
    - if all modes unavailable -> returns NaN
    """
    # scale
    U = np.stack([U1, U2, U3, U4], axis=0) / theta
    U = np.where(np.isfinite(U), U, -np.inf)
    m = np.max(U, axis=0)
    # if all -inf -> m=-inf
    sumexp = np.exp(U - m)  # exp(-inf - -inf) becomes exp(nan); fix below
    # fix positions where m=-inf
    bad = ~np.isfinite(m)
    # for bad positions set sumexp=0
    sumexp[:, bad] = 0.0
    s = np.sum(sumexp, axis=0)
    out = m + np.log(s, where=(s > 0), out=np.full_like(m, np.nan))
    out[bad] = np.nan
    return out * theta

# ------------------------------------------------------------
# Load mapping NO (idx->zone_id) from car ready file
# ------------------------------------------------------------
fcar = omx.open_file(str(CAR_READY), "r")
no_arr = read_no_array_safe(fcar, MAPPING_NAME)
K = len(no_arr)
print("K (zones):", K, "| NO first:", no_arr[:3], "| NO last:", no_arr[-3:])
fcar.close()

# ------------------------------------------------------------
# Load zone attributes needed for utilities (acc_2023)
# at_car (sec), pc_car (CHF/h)
# ------------------------------------------------------------
acc23 = gpd.read_file(ACC2023_GPKG, layer=ACC2023_LAYER)
if acc23.crs is not None and acc23.crs.to_epsg() != 2056:
    acc23 = acc23.to_crs(2056)

acc23 = acc23.set_index(acc23["zone_id"].astype(int))

# arrays aligned to NO order
at_car_sec = acc23.reindex(no_arr)["at_car"].to_numpy(dtype=np.float64)   # seconds
pc_car_chf = acc23.reindex(no_arr)["pc_car"].to_numpy(dtype=np.float64)   # CHF/h

# sanity
print("Missing at_car:", np.isnan(at_car_sec).sum(), "Missing pc_car:", np.isnan(pc_car_chf).sum())

# precompute destination parking term (2h)
park_term = pc_car_chf * 2.0  # CHF

# ------------------------------------------------------------
# Open input skims (ready)
# ------------------------------------------------------------
car = omx.open_file(str(CAR_READY), "r")
pt  = omx.open_file(str(PT_READY), "r")

# car matrices (km/min already)
M_am_dist = car["am_distances_ready"]
M_pm_dist = car["pm_distances_ready"]
M_am_tt   = car["am_travel_times_ready"]
M_pm_tt   = car["pm_travel_times_ready"]

# pt matrices (km/min for distances/time; others as-is)
M_pt_tt     = pt["travel_times_ready"]              # min
M_pt_acc    = pt["access_times_ready"]              # min
M_pt_egr    = pt["egress_times_ready"]              # min
M_pt_freq   = pt["frequencies_ready"]               # departures/hour
M_pt_tr     = pt["transfer_counts_ready"]           # transfers
M_pt_share  = pt["train_traveltime_shares_ready"]   # share in [0,1]

# ------------------------------------------------------------
# Create output OMX utilities file
# ------------------------------------------------------------
if OUT_UTIL.exists():
    OUT_UTIL.unlink()

fout = omx.open_file(str(OUT_UTIL), "w")
# mapping NO as array idx->zone_id
fout.create_mapping(MAPPING_NAME, no_arr.astype(np.int64))

atom_f4 = tb.Float32Atom()

fout.create_matrix("utility_bike", atom=atom_f4, shape=(K, K))
fout.create_matrix("utility_car",  atom=atom_f4, shape=(K, K))
fout.create_matrix("utility_pt",   atom=atom_f4, shape=(K, K))
fout.create_matrix("utility_walk", atom=atom_f4, shape=(K, K))
fout.create_matrix("utility_emu",  atom=atom_f4, shape=(K, K))

U_bike_out = fout["utility_bike"]
U_car_out  = fout["utility_car"]
U_pt_out   = fout["utility_pt"]
U_walk_out = fout["utility_walk"]
EMU_out    = fout["utility_emu"]

# ------------------------------------------------------------
# Main loop: compute utilities row-block by row-block
# ------------------------------------------------------------
for r0 in range(0, K, CHUNK_ROWS):
    r1 = min(K, r0 + CHUNK_ROWS)
    print(f"Computing rows {r0}..{r1-1}")

    # --- Read car skim blocks (safe slicing)
    # dist in km; TT in min
    dist_car = np.asarray(M_pm_dist[r0:r1, :], dtype=np.float64)  # (B,K)
    tt_am    = np.asarray(M_am_tt[r0:r1, :], dtype=np.float64)
    tt_pm    = np.asarray(M_pm_tt[r0:r1, :], dtype=np.float64)
    tt_car   = np.maximum(tt_am, tt_pm)

    # --- distance bands
    d0015 = dist_band_0015(dist_car)
    d1550 = dist_band_1550(dist_car)
    d5099 = dist_band_5099(dist_car)
    d100x = dist_band_100x(dist_car)

    # --- access time term for car: (at_i + at_j)/60 in minutes
    # at_car_sec is 1D aligned to columns; for origin rows we slice
    acc_i = at_car_sec[r0:r1] / 60.0  # minutes, shape (B,)
    acc_j = at_car_sec / 60.0         # minutes, shape (K,)

    acc_term = acc_i[:, None] + acc_j[None, :]  # (B,K)

    # --- parking term uses destination only (K,)
    park = park_term[None, :]  # (1,K)

    # ---- U(bike), U(walk)
    # as given: dist_car / 0.21667 and /0.078336
    U_bike = -0.25 + (-0.150) * (dist_car / 0.21667)
    U_walk = +2.30 + (-0.100) * (dist_car / 0.078336)

    # ---- U(car)
    U_car = (-0.40
             + (-0.053) * tt_car
             + (-0.040) * d0015
             + (-0.040) * d1550
             + ( 0.015) * d5099
             + ( 0.010) * d100x
             + (-0.047) * acc_term
             + (-0.135) * park)

    # --- Read PT skim blocks
    pt_tt   = np.asarray(M_pt_tt[r0:r1, :], dtype=np.float64)      # min
    pt_acc  = np.asarray(M_pt_acc[r0:r1, :], dtype=np.float64)     # min
    pt_egr  = np.asarray(M_pt_egr[r0:r1, :], dtype=np.float64)     # min
    pt_freq = np.asarray(M_pt_freq[r0:r1, :], dtype=np.float64)    # dep/h
    pt_tr   = np.asarray(M_pt_tr[r0:r1, :], dtype=np.float64)      # transfers
    share   = np.asarray(M_pt_share[r0:r1, :], dtype=np.float64)   # share

    # clean: any inf -> nan
    for A in (pt_tt, pt_acc, pt_egr, pt_freq, pt_tr, share):
        A[np.isinf(A)] = np.nan

    # clamp share to [0,1]
    share = np.clip(share, 0.0, 1.0)

    # split into bus/train times
    TT_train = share * pt_tt
    TT_bus   = (1.0 - share) * pt_tt

    # headway term: 60/pt_freq (minutes)
    headway = np.where((pt_freq > 0) & np.isfinite(pt_freq), 60.0 / pt_freq, np.nan)

    # ---- U(pt)  (use distance bands from car distance as in spec)
    U_pt = (+0.75
            + (-0.042)  * TT_bus
            + (-0.0378) * TT_train
            + (-0.015)  * d0015
            + (-0.015)  * d1550
            + ( 0.005)  * d5099
            + ( 0.025)  * d100x
            + (-0.050)  * (pt_acc + pt_egr)
            + (-0.014)  * headway
            + (-0.227)  * pt_tr)

    # ---- EMU logsum (theta_mode=1)
    EMU = logsumexp_4(U_walk, U_bike, U_car, U_pt, theta=1.0)

    # write to output (float32)
    U_bike_out[r0:r1, :] = U_bike.astype(np.float32)
    U_walk_out[r0:r1, :] = U_walk.astype(np.float32)
    U_car_out[r0:r1, :]  = U_car.astype(np.float32)
    U_pt_out[r0:r1, :]   = U_pt.astype(np.float32)
    EMU_out[r0:r1, :]    = EMU.astype(np.float32)

# close files
car.close()
pt.close()
fout.close()

print("\n✅ Done. Utilities written to:", OUT_UTIL)


## Chunk 3

In [ ]:
import numpy as np
from pathlib import Path
import openmatrix as omx

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
UTIL_OMX   = READY_DIR / "utilities_2023_ready.omx"
BEE_OMX    = READY_DIR / "beeline_2023_ready.omx"

UTIL_MATS = ["utility_walk", "utility_bike", "utility_car", "utility_pt", "utility_emu"]
BEE_MAT   = "beeline_km_ready"
MAP_NAME  = "NO"

CHUNK = 200  # adjust if you want faster/slower

def read_no_array_safe(f, name="NO"):
    m = f.mapping(name)
    try:
        return np.asarray(m[:]).astype(int)
    except Exception:
        try:
            return np.asarray(list(m)).astype(int)
        except Exception:
            arr = np.asarray(m)
            if arr.ndim == 0:
                raise ValueError("Mapping NO is scalar/unsized.")
            return arr.astype(int)

def explore_utilities(omx_path):
    f = omx.open_file(str(omx_path), "r")
    print("\n" + "="*90)
    print("READY OMX:", omx_path)
    print("="*90)
    print("Matrices:", list(f.list_matrices()))
    print("Mappings:", list(f.list_mappings()))

    no = read_no_array_safe(f, MAP_NAME)
    print("\nNO mapping (idx -> zone_id):")
    print(" - len:", len(no))
    print(" - first 5:", no[:5].tolist())
    print(" - last 5:", no[-5:].tolist())
    print(" - unique:", len(np.unique(no)), "duplicates?", len(np.unique(no)) != len(no))

    # quick stats per matrix (block + scalar samples)
    rng = np.random.default_rng(0)
    for mn in UTIL_MATS:
        if mn not in f.list_matrices():
            print(f"\n[{mn}] MISSING")
            continue
        M = f[mn]
        n = int(M.shape[0])
        print(f"\n[{mn}] shape={M.shape} dtype={M.dtype}")

        # block
        bs = min(250, n)
        r0 = int(rng.integers(0, max(1, n-bs+1)))
        c0 = int(rng.integers(0, max(1, n-bs+1)))
        block = np.asarray(M[r0:r0+bs, c0:c0+bs], dtype=np.float64)
        finite = block[np.isfinite(block)]
        if finite.size:
            print(f"  block stats: min={finite.min():.3f} mean={finite.mean():.3f} max={finite.max():.3f}")
        else:
            print("  block stats: no finite values")

        # scalar sample
        ii = rng.integers(0, n, size=2000, dtype=np.int64)
        jj = rng.integers(0, n, size=2000, dtype=np.int64)
        vals = np.array([float(M[int(i), int(j)]) for i, j in zip(ii, jj)], dtype=np.float64)
        vals = vals[np.isfinite(vals)]
        if vals.size:
            print(f"  random stats (n={vals.size}): min={vals.min():.3f} mean={vals.mean():.3f} max={vals.max():.3f}")
        else:
            print("  random stats: no finite values")

    f.close()

def count_nans_in_matrix(omx_path, matrix_name, chunk=200):
    f = omx.open_file(str(omx_path), "r")
    M = f[matrix_name]
    n = int(M.shape[0])

    nan_count = 0
    finite_count = 0

    for r0 in range(0, n, chunk):
        r1 = min(n, r0 + chunk)
        block = np.asarray(M[r0:r1, :], dtype=np.float64)
        nan_count += int(np.isnan(block).sum())
        finite_count += int(np.isfinite(block).sum())

    f.close()
    total = n * n
    return {
        "matrix": matrix_name,
        "n": n,
        "total_cells": total,
        "nan": nan_count,
        "finite": finite_count,
        "nan_share": nan_count / total
    }

def sample_corr_emu_beeline(util_path, bee_path, n_samples=200000, seed=0, use_log=False):
    """
    Sample-based correlation (fast, robust):
    - Pearson + Spearman on sampled OD pairs
    - handles NaN/inf
    - optionally log-transform beeline (log1p)
    """
    fu = omx.open_file(str(util_path), "r")
    fb = omx.open_file(str(bee_path), "r")

    EMU = fu["utility_emu"]
    BEE = fb[BEE_MAT]

    n = int(EMU.shape[0])
    rng = np.random.default_rng(seed)

    # sample scalar reads (OMX supports scalar access)
    ii = rng.integers(0, n, size=n_samples, dtype=np.int64)
    jj = rng.integers(0, n, size=n_samples, dtype=np.int64)

    emu_vals = np.empty(n_samples, dtype=np.float64)
    bee_vals = np.empty(n_samples, dtype=np.float64)

    for k, (i, j) in enumerate(zip(ii, jj)):
        emu_vals[k] = float(EMU[int(i), int(j)])
        bee_vals[k] = float(BEE[int(i), int(j)])

    # clean
    good = np.isfinite(emu_vals) & np.isfinite(bee_vals)
    emu_vals = emu_vals[good]
    bee_vals = bee_vals[good]

    if use_log:
        # log(1 + d) to avoid log(0)
        bee_vals = np.log1p(bee_vals)

    # Pearson
    pearson = np.corrcoef(emu_vals, bee_vals)[0, 1] if len(emu_vals) > 2 else np.nan

    # Spearman (rank corr)
    # (fast enough for sample size 200k)
    emu_rank = emu_vals.argsort().argsort().astype(np.float64)
    bee_rank = bee_vals.argsort().argsort().astype(np.float64)
    spearman = np.corrcoef(emu_rank, bee_rank)[0, 1] if len(emu_vals) > 2 else np.nan

    fu.close()
    fb.close()

    return {
        "n_samples_requested": n_samples,
        "n_samples_used": int(len(emu_vals)),
        "pearson": float(pearson),
        "spearman": float(spearman),
        "log_beeline": bool(use_log)
    }

# -------------------------
# RUN
# -------------------------
assert UTIL_OMX.exists(), f"Missing {UTIL_OMX}. Create utilities_2023_ready.omx first."
assert BEE_OMX.exists(),  f"Missing {BEE_OMX}. Create beeline_2023_ready.omx first."

# 1) exploration of utilities
explore_utilities(UTIL_OMX)

# 2) NaN counts (EMU + optionally the 4 mode utilities)
print("\n" + "="*90)
print("NaN COUNTS")
print("="*90)

emu_nan = count_nans_in_matrix(UTIL_OMX, "utility_emu", chunk=CHUNK)
print(emu_nan)

for mn in ["utility_walk", "utility_bike", "utility_car", "utility_pt"]:
    info = count_nans_in_matrix(UTIL_OMX, mn, chunk=CHUNK)
    print(info)

# 3) correlation EMU vs beeline (sample-based)
print("\n" + "="*90)
print("CORRELATION: EMU vs BEELINE")
print("="*90)

corr_lin = sample_corr_emu_beeline(UTIL_OMX, BEE_OMX, n_samples=10, seed=0, use_log=False)
corr_log = sample_corr_emu_beeline(UTIL_OMX, BEE_OMX, n_samples=10, seed=0, use_log=True)

print("Linear beeline:", corr_lin)
print("Log(1+beeline):", corr_log)


## Chunk 4

In [ ]:
import numpy as np
import geopandas as gpd
from pathlib import Path
import openmatrix as omx
import tables as tb

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
CAR_READY = READY_DIR / "car_skims_2023_ready.omx"
PT_READY  = READY_DIR / "pt_skims_2023_ready.omx"
OUT_UTIL_2017 = READY_DIR / "utilities_2017_ready.omx"

ACC2017_GPKG  = Path("TravelCost/accessibility/2017/acc_2017.gpkg")
ACC2017_LAYER = "acc_2017"

MAPPING_NAME = "NO"
CHUNK_ROWS = 200

def read_no_array_safe(f, name="NO"):
    m = f.mapping(name)
    try:
        return np.asarray(m[:]).astype(int)
    except Exception:
        return np.asarray(list(m)).astype(int)

def dist_band_0015(d): return np.minimum(d, 15.0)
def dist_band_1550(d): return np.maximum(0.0, np.minimum(d - 15.0, 35.0))
def dist_band_5099(d): return np.maximum(0.0, np.minimum(d - 50.0, 50.0))
def dist_band_100x(d): return np.maximum(0.0, d - 100.0)

def logsumexp_4(U1, U2, U3, U4, theta=1.0):
    U = np.stack([U1, U2, U3, U4], axis=0) / theta
    U = np.where(np.isfinite(U), U, -np.inf)
    m = np.max(U, axis=0)
    bad = ~np.isfinite(m)
    sumexp = np.exp(U - m)
    sumexp[:, bad] = 0.0
    s = np.sum(sumexp, axis=0)
    out = m + np.log(s, where=(s > 0), out=np.full_like(m, np.nan))
    out[bad] = np.nan
    return out * theta

# ---- mapping NO (idx->zone_id) from CAR_READY
fcar = omx.open_file(str(CAR_READY), "r")
no_arr = read_no_array_safe(fcar, MAPPING_NAME)
K = len(no_arr)
fcar.close()

# ---- load acc_2017 attributes aligned to NO order
acc17 = gpd.read_file(ACC2017_GPKG, layer=ACC2017_LAYER)
acc17 = acc17.set_index(acc17["zone_id"].astype(int))

at_car_sec = acc17.reindex(no_arr)["at_car"].to_numpy(dtype=np.float64)
pc_car_chf = acc17.reindex(no_arr)["pc_car"].to_numpy(dtype=np.float64)

# parking term (2h)
park_term = pc_car_chf * 2.0

# ---- open skims
car = omx.open_file(str(CAR_READY), "r")
pt  = omx.open_file(str(PT_READY), "r")

M_pm_dist = car["pm_distances_ready"]
M_am_tt   = car["am_travel_times_ready"]
M_pm_tt   = car["pm_travel_times_ready"]

M_pt_tt    = pt["travel_times_ready"]
M_pt_acc   = pt["access_times_ready"]
M_pt_egr   = pt["egress_times_ready"]
M_pt_freq  = pt["frequencies_ready"]
M_pt_tr    = pt["transfer_counts_ready"]
M_pt_share = pt["train_traveltime_shares_ready"]

# ---- output
if OUT_UTIL_2017.exists():
    OUT_UTIL_2017.unlink()

fout = omx.open_file(str(OUT_UTIL_2017), "w")
fout.create_mapping(MAPPING_NAME, no_arr.astype(np.int64))

atom = tb.Float32Atom()
for name in ["utility_walk","utility_bike","utility_car","utility_pt","utility_emu"]:
    fout.create_matrix(name, atom=atom, shape=(K, K))

Uwalk_out = fout["utility_walk"]
Ubike_out = fout["utility_bike"]
Ucar_out  = fout["utility_car"]
Upt_out   = fout["utility_pt"]
EMU_out   = fout["utility_emu"]

for r0 in range(0, K, CHUNK_ROWS):
    r1 = min(K, r0 + CHUNK_ROWS)
    print(f"2017 utilities rows {r0}..{r1-1}")

    dist_car = np.asarray(M_pm_dist[r0:r1, :], dtype=np.float64)
    tt_car = np.maximum(
        np.asarray(M_am_tt[r0:r1, :], dtype=np.float64),
        np.asarray(M_pm_tt[r0:r1, :], dtype=np.float64),
    )

    d0015 = dist_band_0015(dist_car)
    d1550 = dist_band_1550(dist_car)
    d5099 = dist_band_5099(dist_car)
    d100x = dist_band_100x(dist_car)

    acc_i = at_car_sec[r0:r1] / 60.0
    acc_j = at_car_sec / 60.0
    acc_term = acc_i[:, None] + acc_j[None, :]

    park = park_term[None, :]

    U_bike = -0.25 + (-0.150) * (dist_car / 0.21667)
    U_walk = +2.30 + (-0.100) * (dist_car / 0.078336)

    U_car = (-0.40
             + (-0.053) * tt_car
             + (-0.040) * d0015
             + (-0.040) * d1550
             + ( 0.015) * d5099
             + ( 0.010) * d100x
             + (-0.047) * acc_term
             + (-0.135) * park)

    pt_tt   = np.asarray(M_pt_tt[r0:r1, :], dtype=np.float64)
    pt_acc  = np.asarray(M_pt_acc[r0:r1, :], dtype=np.float64)
    pt_egr  = np.asarray(M_pt_egr[r0:r1, :], dtype=np.float64)
    pt_freq = np.asarray(M_pt_freq[r0:r1, :], dtype=np.float64)
    pt_tr   = np.asarray(M_pt_tr[r0:r1, :], dtype=np.float64)
    share   = np.asarray(M_pt_share[r0:r1, :], dtype=np.float64)

    for A in (pt_tt, pt_acc, pt_egr, pt_freq, pt_tr, share):
        A[np.isinf(A)] = np.nan
    share = np.clip(share, 0.0, 1.0)

    TT_train = share * pt_tt
    TT_bus   = (1.0 - share) * pt_tt
    headway  = np.where((pt_freq > 0) & np.isfinite(pt_freq), 60.0 / pt_freq, np.nan)

    U_pt = (+0.75
            + (-0.042)  * TT_bus
            + (-0.0378) * TT_train
            + (-0.015)  * d0015
            + (-0.015)  * d1550
            + ( 0.005)  * d5099
            + ( 0.025)  * d100x
            + (-0.050)  * (pt_acc + pt_egr)
            + (-0.014)  * headway
            + (-0.227)  * pt_tr)

    EMU = logsumexp_4(U_walk, U_bike, U_car, U_pt, theta=1.0)

    Ubike_out[r0:r1, :] = U_bike.astype(np.float32)
    Uwalk_out[r0:r1, :] = U_walk.astype(np.float32)
    Ucar_out[r0:r1, :]  = U_car.astype(np.float32)
    Upt_out[r0:r1, :]   = U_pt.astype(np.float32)
    EMU_out[r0:r1, :]   = EMU.astype(np.float32)

car.close(); pt.close(); fout.close()
print("✅ utilities_2017_ready.omx written:", OUT_UTIL_2017)


import numpy as np
from pathlib import Path
import openmatrix as omx

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
UTIL_2017 = READY_DIR / "utilities_2017_ready.omx"
UTIL_2023 = READY_DIR / "utilities_2023_ready.omx"
BEE_OMX   = READY_DIR / "beeline_2023_ready.omx"

def sample_corr_mats(omxA, matA, omxB, matB, n_samples=200000, seed=0, transformB=None):
    fa = omx.open_file(str(omxA), "r")
    fb = omx.open_file(str(omxB), "r")
    A = fa[matA]
    B = fb[matB]
    n = int(A.shape[0])

    rng = np.random.default_rng(seed)
    ii = rng.integers(0, n, size=n_samples, dtype=np.int64)
    jj = rng.integers(0, n, size=n_samples, dtype=np.int64)

    a = np.empty(n_samples, dtype=np.float64)
    b = np.empty(n_samples, dtype=np.float64)
    for k, (i, j) in enumerate(zip(ii, jj)):
        a[k] = float(A[int(i), int(j)])
        b[k] = float(B[int(i), int(j)])

    good = np.isfinite(a) & np.isfinite(b)
    a = a[good]; b = b[good]

    if transformB == "log1p":
        b = np.log1p(b)

    pearson = np.corrcoef(a, b)[0, 1]
    # spearman via ranks
    ar = a.argsort().argsort().astype(np.float64)
    br = b.argsort().argsort().astype(np.float64)
    spearman = np.corrcoef(ar, br)[0, 1]

    fa.close(); fb.close()
    return {"n_used": int(len(a)), "pearson": float(pearson), "spearman": float(spearman), "transformB": transformB}

print("EMU2017 vs EMU2023:",
      sample_corr_mats(UTIL_2017, "utility_emu", UTIL_2023, "utility_emu", n_samples=10, seed=0))

print("EMU2017 vs beeline (linear):",
      sample_corr_mats(UTIL_2017, "utility_emu", BEE_OMX, "beeline_km_ready", n_samples=10, seed=0, transformB=None))

print("EMU2017 vs log1p(beeline):",
      sample_corr_mats(UTIL_2017, "utility_emu", BEE_OMX, "beeline_km_ready", n_samples=10, seed=0, transformB="log1p"))


## Chunk 5

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

TRIPS_DIR = Path("Trips")
files = [
    TRIPS_DIR / "destinations_YS_emu.csv",
    TRIPS_DIR / "destinations_OS_emu.csv",
    TRIPS_DIR / "destinations_YL_emu.csv",
    TRIPS_DIR / "destinations_OL_emu.csv",
]

def pearson_spearman(x, y):
    good = np.isfinite(x) & np.isfinite(y)
    x = x[good]
    y = y[good]

    pear = float(np.corrcoef(x, y)[0, 1]) if len(x) > 2 else np.nan

    # simple rank transform for Spearman
    xr = x.argsort().argsort().astype(np.float64)
    yr = y.argsort().argsort().astype(np.float64)
    spear = float(np.corrcoef(xr, yr)[0, 1]) if len(x) > 2 else np.nan

    return len(x), pear, spear

# Load + concat
dfs = []
for fp in files:
    if not fp.exists():
        print("Missing:", fp)
        continue
    df = pd.read_csv(fp)
    dfs.append(df)

trips = pd.concat(dfs, ignore_index=True)
print("Total trips rows before filtering:", len(trips))

# Exclude NK leisure category
if "leisure_cat" in trips.columns:
    trips["leisure_cat"] = trips["leisure_cat"].astype(str).str.strip()
    trips = trips[trips["leisure_cat"] != "NK"].copy()
else:
    raise KeyError("Column 'leisure_cat' not found in trips data.")

print("Total trips rows after excluding leisure_cat == 'NK':", len(trips))

# Ensure numeric
trips["dist_km"] = pd.to_numeric(trips["dist_km"], errors="coerce")
trips["emu_2017"] = pd.to_numeric(trips["emu_2017"], errors="coerce")
trips["emu_2023"] = pd.to_numeric(trips["emu_2023"], errors="coerce")

# Remove weird distances
trips.loc[trips["dist_km"] < 0, "dist_km"] = np.nan

# Correlations: linear distance
n, p, s = pearson_spearman(
    trips["dist_km"].to_numpy(),
    trips["emu_2017"].to_numpy()
)
print(f"\n[dist_km vs emu_2017, excluding NK] n={n} pearson={p:.4f} spearman={s:.4f}")

n, p, s = pearson_spearman(
    trips["dist_km"].to_numpy(),
    trips["emu_2023"].to_numpy()
)
print(f"[dist_km vs emu_2023, excluding NK] n={n} pearson={p:.4f} spearman={s:.4f}")

# Correlations: log(1+dist)
logd = np.log1p(trips["dist_km"].to_numpy())

n, p, s = pearson_spearman(
    logd,
    trips["emu_2017"].to_numpy()
)
print(f"\n[log1p(dist_km) vs emu_2017, excluding NK] n={n} pearson={p:.4f} spearman={s:.4f}")

n, p, s = pearson_spearman(
    logd,
    trips["emu_2023"].to_numpy()
)
print(f"[log1p(dist_km) vs emu_2023, excluding NK] n={n} pearson={p:.4f} spearman={s:.4f}")

## Chunk 6

In [ ]:
# ============================================================
# Create utilities_2023_ready_woLIE.omx AND beeline_2023_ready_woLIE.omx
# by removing Liechtenstein zones from mapping "NO"
# - drops rows/cols for those zone IDs
# - writes filtered mapping "NO"
# - filters ALL square NxN matrices that match mapping size (safe default)
# - copies non-matching matrices as-is
# ============================================================

import numpy as np
from pathlib import Path
import openmatrix as omx

# -----------------------------
# CONFIG
# -----------------------------
READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
MAP_NAME = "NO"

UTIL_IN  = READY_DIR / "utilities_2023_ready.omx"
UTIL_OUT = READY_DIR / "utilities_2023_ready_woLIE.omx"

BEE_IN   = READY_DIR / "beeline_2023_ready.omx"
BEE_OUT  = READY_DIR / "beeline_2023_ready_woLIE.omx"

LIE_NPVM = {
    700101001, 700201001, 700301001, 700401001, 700501001, 700601001, 700701001,
    700801001, 700901001, 701001001, 701101001, 710101001, 730101001
}

assert UTIL_IN.exists(), f"Missing input OMX: {UTIL_IN}"
assert BEE_IN.exists(),  f"Missing input OMX: {BEE_IN}"

# -----------------------------
# Helper: filter one OMX
# -----------------------------
def filter_omx_woLIE(in_path: Path, out_path: Path, map_name: str):
    fin = omx.open_file(str(in_path), "r")

    # Read mapping NO (idx -> zone_id)
    m = fin.mapping(map_name)
    try:
        no = np.asarray(m[:]).astype(int)
    except Exception:
        no = np.asarray(list(m)).astype(int)

    N = len(no)
    print(f"\n[IN] {in_path.name} | mapping '{map_name}' size={N}")

    # Build keep mask
    keep = np.array([z not in LIE_NPVM for z in no], dtype=bool)
    keep_idx = np.where(keep)[0]
    drop_idx = np.where(~keep)[0]

    print(f"[woLIE] dropping {len(drop_idx)} zones, keeping {len(keep_idx)}")

    no_keep = no[keep_idx].astype(int)

    # Write output
    fout = omx.open_file(str(out_path), "w")

    # Write filtered mapping
    fout.create_mapping(map_name, no_keep.tolist())

    # Copy & filter all matrices
    mats = list(fin.list_matrices())
    print(f"[IN] matrices found: {len(mats)}")
    for name in mats:
        M = fin[name]  # OMXMatrix

        # Filter only if square NxN matching mapping size
        if len(M.shape) != 2 or M.shape[0] != N or M.shape[1] != N:
            print(f"[SKIP-FILTER] {name}: shape={M.shape} (does not match mapping size), copying as-is")
            data = np.asarray(M[:, :])
            fout.create_matrix(name, obj=data)
            continue

        print(f"[FILTER] {name}")
        data = np.asarray(M[:, :], dtype=np.float32)  # load once
        data_wo = data[np.ix_(keep_idx, keep_idx)]
        fout.create_matrix(name, obj=data_wo)

    # Copy file attributes (best-effort)
    try:
        attrs = fin.list_attributes()
        # some openmatrix versions return dict-like, some list; handle dict-like
        if hasattr(attrs, "items"):
            for k, v in attrs.items():
                fout.create_attribute(k, v)
    except Exception:
        pass

    fin.close()
    fout.close()
    print(f"[OK] wrote: {out_path}")

# -----------------------------
# RUN both OMX files
# -----------------------------
filter_omx_woLIE(UTIL_IN, UTIL_OUT, MAP_NAME)
filter_omx_woLIE(BEE_IN,  BEE_OUT,  MAP_NAME)

print("\n[DONE]")


## Chunk 7

In [ ]:
from pathlib import Path
import numpy as np
import openmatrix as omx
import tables as tb

# --------------------------------------------------
# PATHS
# --------------------------------------------------
READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")

CAR_OMX = READY_DIR / "car_skims_2023_ready.omx"
OUT_OMX = READY_DIR / "distance_avg_2023_ready.omx"

AM_NAME = "am_distances_ready"
PM_NAME = "pm_distances_ready"
OUT_NAME = "avg_distances_ready"
MAP_NAME = "NO"

ROW_BLOCK = 512  # increase if RAM allows


# --------------------------------------------------
# HELPERS
# --------------------------------------------------
def read_mapping_safe(f, name="NO"):
    m = f.mapping(name)
    try:
        arr = np.asarray(m[:]).astype(int)
        if arr.ndim == 1:
            return arr
    except Exception:
        pass
    try:
        arr = np.asarray(list(m)).astype(int)
        if arr.ndim == 1:
            return arr
    except Exception:
        pass
    arr = np.asarray(m).astype(int)
    if arr.ndim != 1:
        raise ValueError(f"Mapping {name} is not 1D.")
    return arr


# --------------------------------------------------
# MAIN
# --------------------------------------------------
assert CAR_OMX.exists(), f"Missing file: {CAR_OMX}"

fin = omx.open_file(str(CAR_OMX), "r")

try:
    mats = set(fin.list_matrices())
    assert AM_NAME in mats, f"{AM_NAME} not found in {CAR_OMX}"
    assert PM_NAME in mats, f"{PM_NAME} not found in {CAR_OMX}"

    no_arr = read_mapping_safe(fin, MAP_NAME)

    am_shape = fin[AM_NAME].shape
    pm_shape = fin[PM_NAME].shape
    assert am_shape == pm_shape, f"Shape mismatch: AM={am_shape}, PM={pm_shape}"

    K = am_shape[0]
    print(f"K zones: {K}")
    print(f"AM shape: {am_shape}")
    print(f"PM shape: {pm_shape}")

    if OUT_OMX.exists():
        OUT_OMX.unlink()

    fout = omx.open_file(str(OUT_OMX), "w")
    try:
        fout.create_mapping(MAP_NAME, no_arr.astype(np.int64))
    except Exception:
        fout.create_mapping(MAP_NAME, [int(x) for x in no_arr.tolist()])

    atom_f4 = tb.Float32Atom()
    fout.create_matrix(OUT_NAME, atom=atom_f4, shape=(K, K))
    out_mat = fout[OUT_NAME]

    for r0 in range(0, K, ROW_BLOCK):
        r1 = min(K, r0 + ROW_BLOCK)
        print(f"Processing rows {r0}..{r1-1}")

        am_blk = np.asarray(fin[AM_NAME][r0:r1, :], dtype=np.float32)
        pm_blk = np.asarray(fin[PM_NAME][r0:r1, :], dtype=np.float32)

        avg_blk = 0.5 * (am_blk + pm_blk)
        out_mat[r0:r1, :] = avg_blk.astype(np.float32)

    print(f"\n✅ Done. Wrote average distance matrix to: {OUT_OMX}")
    print(f"Matrix name: {OUT_NAME}")

    fout.close()

finally:
    fin.close()

## Chunk 8

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import openmatrix as omx
from pathlib import Path

# =========================
# CONFIG
# =========================
TZ_GPKG   = "VariableAnalysis/TZ_first_sel_log1p.gpkg"
TZ_LAYER  = "TZ_first_sel_log1p"
TZ_JOIN_KEY = "npvm_id"

READY_DIR = Path("TravelCost/skims/MOBi5_2023_ready")
UTIL_OMX      = READY_DIR / "utilities_2023_ready.omx"
EMU_MAT_NAME  = "utility_emu"
MAP_NAME      = "NO"

OUT_CSV = "ModelRuns/BaselineFinal2/corr_accessibility_vs_factors.csv"

# Final 13 variables in thesis order
VARS_13 = [
    ("F1",    "F1_gastr_count_log1p"),
    ("F2",    "F2_pop_total_log1p"),
    ("F3",    "F8_outdoor_lake_raw_dens_log1p"),
    ("F3",    "F3_outdoor_hard_count_log1p"),
    ("F3",    "F3_outdoor_soft_count_log1p"),
    ("F3/F7", "F3_outdoor_LUmix"),
    ("F4",    "F4_cult_count_log1p"),
    ("F5",    "F5_sport_count_log1p"),
    ("F5/F3", "F5_sport_out_length_log1p"),
    ("F6",    "F6_others_count_log1p"),
    ("F7",    "F7_urban_sum_log1p"),
    ("F8",    "F8_POI_urban_dens_log1p"),
    ("F10",   "F10_superinfra_log1p"),
]

# =========================
# HELPERS
# =========================
def load_matrix_and_mapping(omx_path: Path, mat_name: str, map_name: str):
    assert omx_path.exists(), f"Missing OMX: {omx_path}"
    f = omx.open_file(str(omx_path), "r")

    if mat_name not in f.list_matrices():
        raise ValueError(f"Matrix '{mat_name}' not in {omx_path}. Available: {list(f.list_matrices())}")

    m = f.mapping(map_name)
    try:
        idx_to_zone = np.asarray(m[:]).astype(int)
    except Exception:
        idx_to_zone = np.asarray(list(m)).astype(int)

    zone_to_idx = {int(z): int(i) for i, z in enumerate(idx_to_zone)}
    mat = np.asarray(f[mat_name][:, :], dtype=np.float32)
    f.close()
    return mat, zone_to_idx, idx_to_zone

def logsumexp_axis0(X: np.ndarray) -> np.ndarray:
    m = np.max(X, axis=0)
    return m + np.log(np.sum(np.exp(X - m), axis=0) + 1e-30)

# =========================
# LOAD TZ
# =========================
print("[LOAD] TZ layer...")
tz = gpd.read_file(TZ_GPKG, layer=TZ_LAYER)

if TZ_JOIN_KEY not in tz.columns:
    raise ValueError(f"TZ layer missing '{TZ_JOIN_KEY}'")

tz[TZ_JOIN_KEY] = pd.to_numeric(tz[TZ_JOIN_KEY], errors="coerce")
tz = tz.dropna(subset=[TZ_JOIN_KEY]).copy()
tz[TZ_JOIN_KEY] = tz[TZ_JOIN_KEY].astype(int)

missing_vars = [v for _, v in VARS_13 if v not in tz.columns]
if missing_vars:
    raise ValueError(f"Missing variables in TZ layer: {missing_vars}")

# =========================
# LOAD EMU + BUILD DESTINATION-SIDE ACCESSIBILITY
# =========================
print("[LOAD] EMU matrix...")
emu_mat, zone_to_idx, idx_to_zone = load_matrix_and_mapping(UTIL_OMX, EMU_MAT_NAME, MAP_NAME)

print("[AGG] Computing destination-side EMU aggregates...")
acc_df = pd.DataFrame({
    "zone_id": idx_to_zone.astype(int),
    "emu_sum_i": np.sum(emu_mat, axis=0).astype(np.float64),
    "emu_mean_i": np.mean(emu_mat, axis=0).astype(np.float64),
    "emu_logsumexp_i": logsumexp_axis0(emu_mat).astype(np.float64),
})

print("[JOIN] Merging TZ with accessibility aggregates...")
tz2 = tz.merge(acc_df, how="left", left_on=TZ_JOIN_KEY, right_on="zone_id")

# =========================
# CORRELATIONS
# =========================
print("[CORR] Computing Pearson and Spearman correlations...")
rows = []

target_col = "emu_logsumexp_i"   # main accessibility indicator to compare against

for block, var in VARS_13:
    sub = tz2[[var, target_col]].dropna().copy()

    pearson = sub[var].corr(sub[target_col], method="pearson")
    spearman = sub[var].corr(sub[target_col], method="spearman")

    rows.append({
        "Block": block,
        "Variable": var,
        "Pearson_with_EMUlogsum": pearson,
        "Spearman_with_EMUlogsum": spearman,
        "N": len(sub),
    })

corr_df = pd.DataFrame(rows)

# Optional: rounded display version
corr_df_display = corr_df.copy()
corr_df_display["Pearson_with_EMUlogsum"] = corr_df_display["Pearson_with_EMUlogsum"].round(3)
corr_df_display["Spearman_with_EMUlogsum"] = corr_df_display["Spearman_with_EMUlogsum"].round(3)

print("\nCorrelation of final 13 variables with destination-side accessibility (emu_logsumexp_i):\n")
print(corr_df_display.to_string(index=False))

corr_df.to_csv(OUT_CSV, index=False)
print(f"\n[OK] wrote {OUT_CSV}")